In [41]:
import json
import numpy as np
import pandas as pd
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go

raw = np.load(
    "data/external/PITZ_laser_pulse_shaping_v30_features.npz", allow_pickle=True
)
data = raw["features"]
metadata = json.loads(raw["metadata"].item())
data = (
    pd.DataFrame(data, columns=metadata["columns"])
    .astype(metadata["dtypes"])
    .select_dtypes(include=["number"])
)
columns = data.columns
data = data.values
N, D = data.shape

app = dash.Dash(__name__)

app.layout = html.Div(
    [
        html.Div(
            [
                html.Label("X Axis"),
                dcc.Dropdown(
                    options=[{"label": columns[i], "value": i} for i in range(D)],
                    value=0,
                    id="x-dim",
                ),
                html.Label("Y Axis"),
                dcc.Dropdown(
                    options=[{"label": columns[i], "value": i} for i in range(D)],
                    value=1,
                    id="y-dim",
                ),
                html.Label("Z Axis (optional)"),
                dcc.Dropdown(
                    options=[{"label": columns[i], "value": i} for i in range(D)],
                    value=None,
                    id="z-dim",
                ),
                html.Label("Color (optional)"),
                dcc.Dropdown(
                    options=[{"label": columns[i], "value": i} for i in range(D)],
                    value=None,
                    id="color-dim",
                    clearable=True,
                ),
                html.Br(),
                html.Label("X Range"),
                dcc.RangeSlider(id="x-range"),
                html.Label("Y Range"),
                dcc.RangeSlider(id="y-range"),
                html.Label("Z Range"),
                dcc.RangeSlider(id="z-range"),
                html.Label("Color Range"),
                dcc.RangeSlider(id="color-range"),
                html.Br(),
                dcc.Checklist(
                    options=[
                        {"label": "Log X", "value": "logx"},
                        {"label": "Log Y", "value": "logy"},
                        {"label": "Log Z", "value": "logz"},
                    ],
                    value=[],
                    id="log-toggles",
                ),
            ],
            style={"width": "25%", "display": "inline-block"},
        ),
        dcc.Graph(id="scatter-plot", style={"width": "70%", "display": "inline-block"}),
    ]
)


@app.callback(
    Output("scatter-plot", "figure"),
    Output("x-range", "min"),
    Output("x-range", "max"),
    Output("x-range", "value"),
    Output("y-range", "min"),
    Output("y-range", "max"),
    Output("y-range", "value"),
    Output("z-range", "min"),
    Output("z-range", "max"),
    Output("z-range", "value"),
    Output("color-range", "min"),
    Output("color-range", "max"),
    Output("color-range", "value"),
    Input("x-dim", "value"),
    Input("x-range", "value"),
    Input("y-dim", "value"),
    Input("y-range", "value"),
    Input("z-dim", "value"),
    Input("z-range", "value"),
    Input("color-dim", "value"),
    Input("color-range", "value"),
    Input("log-toggles", "value"),
)
def update_plot(
    x_dim,
    x_range,
    y_dim,
    y_range,
    z_dim,
    z_range,
    c_dim,
    c_range,
    log_toggles,
):
    # get data axes
    x = data[:, x_dim]
    y = data[:, y_dim]
    z = data[:, z_dim] if z_dim is not None else None
    c = data[:, c_dim] if c_dim is not None else None

    # initialize ranges if not set
    if x_range is None:
        x_range = [float(np.min(x)), float(np.max(x))]
    if y_range is None:
        y_range = [float(np.min(y)), float(np.max(y))]
    if z_dim is not None and z_range is None:
        z_range = [float(np.min(z)), float(np.max(z))]
    if c_dim is not None and c_range is None:
        c_range = [float(np.min(c)), float(np.max(c))]

    # filter data
    mask = data[:, x_dim] > 0 if "logx" in log_toggles else np.ones_like(x, dtype=bool)
    mask &= data[:, y_dim] > 0 if "logy" in log_toggles else np.ones_like(y, dtype=bool)
    if z_dim is not None and "logz" in log_toggles:
        mask &= data[:, z_dim] > 0
    if c_dim is not None and "logc" in log_toggles:
        mask &= data[:, c_dim] > 0
    mask = (
        mask & (x > x_range[0]) & (x < x_range[1]) & (y > y_range[0]) & (y < y_range[1])
    )
    if z_dim is not None:
        mask &= (z > z_range[0]) & (z < z_range[1])
    if c_dim is not None:
        mask &= (c > c_range[0]) & (c < c_range[1])

    # create plot
    if z_dim is None:
        fig = go.Figure()
        marker = dict(size=6)
        if c_dim is not None:
            marker.update(
                color=c[mask],
                colorscale="Viridis",
                showscale=True,
                colorbar=dict(title=columns[c_dim]),
            )
        fig.add_trace(go.Scatter(x=x[mask], y=y[mask], mode="markers", marker=marker))
        fig.update_layout(
            xaxis=dict(
                title=columns[x_dim],
                type="log" if "logx" in log_toggles else "linear",
            ),
            yaxis=dict(
                title=columns[y_dim],
                type="log" if "logy" in log_toggles else "linear",
            ),
            template="plotly_white",
        )

    elif z_dim is not None:
        fig = go.Figure()
        marker = dict(size=6)
        if c_dim is not None:
            marker.update(
                color=c[mask],
                colorscale="Viridis",
                showscale=True,
                colorbar=dict(title=columns[c_dim]),
            )
        fig.add_trace(
            go.Scatter3d(x=x[mask], y=y[mask], z=z[mask], mode="markers", marker=marker)
        )
        fig.update_layout(
            scene=dict(
                xaxis=dict(
                    title=columns[x_dim],
                    type="log" if "logx" in log_toggles else "linear",
                ),
                yaxis=dict(
                    title=columns[y_dim],
                    type="log" if "logy" in log_toggles else "linear",
                ),
                zaxis=dict(
                    title=columns[z_dim],
                    type="log" if "logz" in log_toggles else "linear",
                ),
            ),
            template="plotly_white",
        )

    return (
        fig,
        float(np.min(x)),
        float(np.max(x)),
        x_range,
        float(np.min(y)),
        float(np.max(y)),
        y_range,
        float(np.min(z)) if z_dim is not None else None,
        float(np.max(z)) if z_dim is not None else None,
        z_range if z_range is not None else None,
        float(np.min(c)) if c is not None else None,
        float(np.max(c)) if c is not None else None,
        c_range if c_range is not None else None,
    )


if __name__ == "__main__":
    app.run(
        jupyter_mode="inline",
        debug=False,
        port=8050,
        use_reloader=False,
    )